# Evaluating Your Judge

## Overview

In notebook 01 we built an LLM-as-Judge evaluator. But how do we know the judge itself is reliable? A judge can sound confident while being wrong. It can reward verbosity over correctness, or flip its verdict on the same input across runs.

This notebook implements a rigorous judge calibration workflow based on the **evaluation-driven philosophy**: every automated evaluator must be validated against human-labeled ground truth before it is trusted in production.

## Key Principles

- **Binary pass/fail only** — no 1-5 scales. Scales introduce implicit variation that hides actual failure modes.
- **One failure mode per judge** — each judge evaluates exactly ONE thing. Combine results afterward.
- **Calibrate against human labels** — split labeled data into few-shot / dev / test sets and iterate.
- **Measure what matters** — True Positive Rate (TPR) and True Negative Rate (TNR) tell you how accurate your judge is as an estimator of real model error.

## What We'll Cover

1. Define a single failure mode with clear pass/fail criteria
2. Build a focused judge prompt with structured output
3. Split human-labeled benchmark data into few-shot / dev / test
4. Calibrate the judge on the dev set and analyze disagreements
5. Iterate on the judge prompt based on failure patterns
6. Validate on the held-out test set
7. Test judge repeatability across multiple runs
8. Produce a judge scorecard

## Prerequisites
- AWS account with Bedrock access (judge model set in `../model_config.py` — Claude Sonnet 5 by default)
- Python 3.10+
- `judge_benchmark.jsonl` in the same directory

## Setup and Dependencies

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import json
import random
import boto3
import pandas as pd
from collections import Counter
from time import sleep
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Bedrock client setup

bedrock = boto3.client('bedrock-runtime')

# Configuration
# Judge model is centralised in ../model_config.py
import sys
sys.path.append("..")
from model_config import JUDGE_MODEL_ID as JUDGE_MODEL
BENCHMARK_PATH = "./judge_benchmark.jsonl"
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print("✅ Setup complete")

## 1. Why Evaluate the Judge?

Consider these real failure modes of LLM judges:

| Judge Failure | What Happens | Consequence |
|---|---|---|
| **Verbosity bias** | Longer responses get "pass" even when wrong | Inflated quality scores |
| **Confidence bias** | Confident-sounding wrong answers get "pass" | Missed hallucinations |
| **Vague rubric** | Judge interprets "good" differently each run | Unstable, unreproducible scores |
| **Multi-criteria overload** | One prompt checks accuracy AND completeness AND tone | Cannot isolate which criterion failed |

The fix is simple: treat your judge like any other model. Give it labeled test data, measure its accuracy, and iterate until it meets your bar.

> *"A vague judge prompt produces vague results."* — Evaluations Prescriptive Guidance

### The Anti-Pattern: Rating Scales

Rating on a scale from 1-5 introduces massive implicit variation. Ask someone to rate on 1-5 vs 3-7 — same scale width, different numbers, different ratings. All of this hides the actual failure state. **Binary pass/fail is far more useful.**

### Think of it like a legal system

If the technical terms (dev set, few-shot split, TPR/TNR) feel abstract, this analogy maps the entire notebook to something familiar:

| Legal System | What we do in this notebook |
|---|---|
| **Congress writes laws** | You write the judge prompt (the rubric with pass/fail criteria) |
| **Laws have grey areas** | Borderline cases emerge (is "920,000" for 918,915 close enough?) |
| **Court cases expose the grey areas** | Dev set disagreements reveal where the rubric is ambiguous |
| **Congress amends the law to clarify** | You refine the prompt (v1 to v2) with explicit edge cases |
| **Case law (precedent) guides future rulings** | Few-shot examples with human rationales |
| **An appeals court checks the lower court** | Test set validation checks the calibrated judge |
| **Judicial consistency (same crime, same sentence)** | Repeatability test (same input, same verdict every time) |

The key insight: **the judge doesn't decide what's right or wrong. It applies rules you wrote.** If it gets a verdict wrong, the fix is almost always in the rules (refining the prompt), not in replacing the judge (swapping the model). You only swap the model after you've exhausted prompt refinement.

Keep this analogy in mind as you work through the sections below. Each step maps to a row in this table.

## 2. Define a Failure Mode

Each judge should evaluate **one specific failure mode**. Not "is this response good?" but "does this response contain the correct factual data from the provided context?"

For this notebook, our failure mode is:

> **Factual Accuracy**: Does the model's response contain population or land area numbers that match the provided context data?

### Pass/Fail Definitions

| Verdict | Definition |
|---|---|
| **Pass** | The numerical data in the response matches the context. Approximate values (e.g., "about 2.4 million" for 2,390,125) are acceptable. Additional commentary that doesn't contradict the data is fine. |
| **Fail** | The response contains a specific but wrong number, reverses a comparison, claims data is unavailable when it's in the context, or doesn't answer the question. |

## 3. Build the Judge Prompt

A good judge prompt has:
- **One criterion** — factual accuracy only
- **Clear pass/fail definitions** — no ambiguity
- **Structured output** — JSON with reasoning before verdict (forces the judge to think before deciding)
- **Few-shot examples** — from the training split (added in Section 5)

### What bad judge prompts look like

```
❌ "Is this response good?"
❌ "Rate this response from 1-5"
❌ "Check accuracy, completeness, tone, and helpfulness. Give pass/fail."
```

Each of these is too vague or too broad.

In [ ]:
# Judge prompt template — ONE failure mode, binary pass/fail, structured output
JUDGE_PROMPT_TEMPLATE = """You are evaluating whether a model's response contains factually accurate data compared to the provided context.

## Evaluation Criterion: Factual Accuracy
- PASS: The numerical data in the response matches the context. Approximate values (e.g., "about 2.4 million" for 2,390,125) are acceptable. Additional commentary that does not contradict the data is fine.
- FAIL: The response contains a specific but wrong number, reverses a comparison, claims data is unavailable when it is in the context, or does not answer the question asked.

{few_shot_section}

## Now evaluate this response:

**Question:** {question}
**Context:** {context}
**Model Response:** {model_response}

Respond with ONLY this JSON (no other text):
{{
    "reasoning": "<your step-by-step analysis>",
    "label": "<pass or fail>"
}}
"""

print("Judge prompt template defined")
print(f"Template length: {len(JUDGE_PROMPT_TEMPLATE)} chars")

## 4. Load and Split the Gold Benchmark

The benchmark file contains human-labeled examples with clear pass/fail verdicts and rationales. We split it into three sets:

| Split | Size | Purpose |
|---|---|---|
| **Few-shot (training)** | ~15% | Examples placed directly in the judge prompt |
| **Dev** | ~45% | Iterative calibration — compare judge to human labels |
| **Test** | ~40% | Final blind validation — never look at during development |

**This is a separate split from your main evaluation dataset.** This small labeled set is specifically for building and validating the judge itself.

### What happens below

We load the benchmark file, then split it into three non-overlapping sets (few-shot, dev, test). The critical detail: we split by **unique question**, not by individual example. Why? Because the same question can appear multiple times with different failure modes (e.g., "What is the population of Dallas?" might have both a factually-wrong response and an incomplete response). If we split naively by row, the same question could land in both the few-shot prompt and the test set. When that happens, the judge isn't independently evaluating the test example; it's pattern-matching against the answer it already saw in-context. Splitting by question prevents this contamination and gives us an honest accuracy measurement.

In [ ]:
# --- Step 1: Load the human-labeled benchmark ---
with open(BENCHMARK_PATH, 'r') as f:
    benchmark = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(benchmark)} benchmark examples")
print(f"Label distribution: {Counter(ex['human_label'] for ex in benchmark)}")

# --- Step 2: Split by QUESTION (not by row) to prevent data leakage ---
# Why by question? The same question can appear with multiple failure modes.
# If "What is the population of Dallas?" ends up in both few-shot and test,
# the judge can pattern-match from the in-context example rather than
# independently evaluating the response, inflating test accuracy.
unique_questions = list(set(ex['question'] for ex in benchmark))
random.shuffle(unique_questions)

# --- Step 3: Allocate questions to splits (15% / 45% / 40%) ---
n_q = len(unique_questions)
n_fewshot_q = max(2, int(n_q * 0.15))  # At least 2 questions for few-shot
n_test_q = int(n_q * 0.40)             # 40% held out, never touched during development

fewshot_questions = set(unique_questions[:n_fewshot_q])
test_questions = set(unique_questions[n_fewshot_q:n_fewshot_q + n_test_q])
dev_questions = set(unique_questions[n_fewshot_q + n_test_q:])  # Remainder goes to dev

# --- Step 4: Assign examples to their split based on which question they belong to ---
fewshot_set = [ex for ex in benchmark if ex['question'] in fewshot_questions]
dev_set = [ex for ex in benchmark if ex['question'] in dev_questions]
test_set = [ex for ex in benchmark if ex['question'] in test_questions]

print(f"\nSplit sizes:")
print(f"  Few-shot (training): {len(fewshot_set)} ({len(fewshot_questions)} unique questions)")
print(f"  Dev (calibration):   {len(dev_set)} ({len(dev_questions)} unique questions)")
print(f"  Test (validation):   {len(test_set)} ({len(test_questions)} unique questions)")

# --- Step 5: Verify no overlap between splits ---
assert not (fewshot_questions & dev_questions), "Data leakage: few-shot overlaps dev"
assert not (fewshot_questions & test_questions), "Data leakage: few-shot overlaps test"
assert not (dev_questions & test_questions), "Data leakage: dev overlaps test"
print("\n✅ No data leakage between splits")

### Benchmark Coverage Analysis

Before proceeding, let's verify our benchmark has adequate coverage across failure modes and label distributions.

In [ ]:
# Benchmark Coverage Analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: examples per failure mode
mode_counts = Counter(ex['failure_mode'] for ex in benchmark)
modes = list(mode_counts.keys())
counts = [mode_counts[m] for m in modes]
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2'][:len(modes)]
axes[0].bar(modes, counts, color=colors)
axes[0].set_ylabel('Number of Examples')
axes[0].set_title('Examples per Failure Mode')
for i, v in enumerate(counts):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

# Right: pass/fail distribution per mode
mode_labels = {}
for ex in benchmark:
    mode = ex['failure_mode']
    label = ex['human_label']
    if mode not in mode_labels:
        mode_labels[mode] = {'pass': 0, 'fail': 0}
    mode_labels[mode][label] += 1

modes_list = list(mode_labels.keys())
pass_counts = [mode_labels[m]['pass'] for m in modes_list]
fail_counts = [mode_labels[m]['fail'] for m in modes_list]

x = np.arange(len(modes_list))
width = 0.35
axes[1].bar(x - width/2, pass_counts, width, label='Pass', color='#55A868')
axes[1].bar(x + width/2, fail_counts, width, label='Fail', color='#C44E52')
axes[1].set_xticks(x)
axes[1].set_xticklabels(modes_list)
axes[1].set_ylabel('Count')
axes[1].set_title('Label Distribution per Failure Mode')
axes[1].legend()

plt.tight_layout()
plt.show()

# Coverage warnings
for mode, count in mode_counts.items():
    if count < 10:
        print(f"⚠️ {mode}: only {count} examples — consider adding more for robust calibration")
    else:
        print(f"✅ {mode}: {count} examples")

## 5. Calibrate the Judge on the Dev Set

First, we build the few-shot section from our training split, then run the judge on every dev example and compare to human labels.

In [ ]:
def build_few_shot_section(examples):
    """Build few-shot examples string from training split."""
    if not examples:
        return ""
    
    lines = ["## Examples for reference:"]
    for i, ex in enumerate(examples, 1):
        lines.append(f"""
### Example {i}:
**Question:** {ex['question']}
**Context:** {ex['context']}
**Model Response:** {ex['model_response']}
**Verdict:** {ex['human_label']}
**Reasoning:** {ex['human_rationale']}""")
    return "\n".join(lines)


def run_judge(question, context, model_response, few_shot_examples):
    """Run the judge on a single example. Returns parsed dict or error."""
    few_shot_section = build_few_shot_section(few_shot_examples)
    
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        question=question,
        context=context,
        model_response=model_response,
        few_shot_section=few_shot_section
    )
    
    response = bedrock.converse(
        modelId=JUDGE_MODEL,
        messages=[{'role': 'user', 'content': [{'text': prompt}]}],
        inferenceConfig={'maxTokens': 1000}
    )
    
    # Safely extract text from response content block
    content_blocks = response.get('output', {}).get('message', {}).get('content', [])
    text = ''
    for block in content_blocks:
        if 'text' in block:
            text = block['text'].strip()
            break
    
    if not text:
        return {"reasoning": "No text in response", "label": "error", "raw": str(content_blocks)}
    
    # Strip markdown fences if present
    if text.startswith("```"):
        lines = text.split("\n")
        lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    
    try:
        result = json.loads(text)
        result['label'] = result['label'].strip().lower()
        return result
    except json.JSONDecodeError:
        return {"reasoning": text, "label": "error", "raw": text}


def evaluate_split(split_data, few_shot_examples, split_name="dev"):
    """Run judge on a full split and compare to human labels."""
    results = []
    
    for i, ex in enumerate(split_data):
        print(f"  Evaluating {split_name} {i+1}/{len(split_data)}...", end="\r")
        
        judge_result = run_judge(
            ex['question'], ex['context'], ex['model_response'], few_shot_examples
        )
        
        results.append({
            'question': ex['question'],
            'model_response': ex['model_response'],
            'human_label': ex['human_label'],
            'judge_label': judge_result['label'],
            'judge_reasoning': judge_result.get('reasoning', ''),
            'human_rationale': ex['human_rationale'],
            'agree': ex['human_label'] == judge_result['label']
        })
        
        sleep(0.5)  # Rate limiting
    
    print(f"  {'':50}")  # Clear progress line
    return results


print("Helper functions defined ✅")

In [ ]:
# Run judge on dev set
print("Running judge on dev set...")
dev_results = evaluate_split(dev_set, fewshot_set, "dev")

# Calculate metrics
df_dev = pd.DataFrame(dev_results)

# True Positive Rate (TPR): Of actual passes, how many did the judge correctly identify?
actual_pass = df_dev[df_dev['human_label'] == 'pass']
tp = len(actual_pass[actual_pass['judge_label'] == 'pass'])
tpr = tp / len(actual_pass) if len(actual_pass) > 0 else 0

# True Negative Rate (TNR): Of actual fails, how many did the judge correctly identify?
actual_fail = df_dev[df_dev['human_label'] == 'fail']
tn = len(actual_fail[actual_fail['judge_label'] == 'fail'])
tnr = tn / len(actual_fail) if len(actual_fail) > 0 else 0

overall_accuracy = df_dev['agree'].mean()

print(f"\n{'='*50}")
print(f"Dev Set Results (n={len(dev_results)})")
print(f"{'='*50}")
print(f"Overall accuracy: {overall_accuracy:.1%}")
print(f"TPR (catches real passes):  {tpr:.1%}  ({tp}/{len(actual_pass)})")
print(f"TNR (catches real fails):   {tnr:.1%}  ({tn}/{len(actual_fail)})")
print(f"{'='*50}")

In [ ]:
# Confusion Matrix — Dev Set
fig, ax = plt.subplots(figsize=(6, 5))

# Build confusion matrix from dev results
cm_labels = ['pass', 'fail']
cm = np.zeros((2, 2), dtype=int)
for _, row in df_dev.iterrows():
    true_idx = 0 if row['human_label'] == 'pass' else 1
    pred_idx = 0 if row['judge_label'] == 'pass' else 1
    cm[true_idx][pred_idx] += 1

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cm_labels, yticklabels=cm_labels, ax=ax)
ax.set_xlabel('Judge Label')
ax.set_ylabel('Human Label')
ax.set_title('Confusion Matrix — Dev Set (v1)')
plt.tight_layout()
plt.show()

print(f"True Positives:  {cm[0][0]}  |  False Negatives: {cm[0][1]}")
print(f"False Positives: {cm[1][0]}  |  True Negatives:  {cm[1][1]}")

## 6. Analyze Disagreements

The most valuable part of calibration is understanding **where the judge disagrees with humans**. These disagreements are the judge's failure modes — and they tell you exactly how to improve the prompt.

In [ ]:
# Show disagreements
disagreements = df_dev[~df_dev['agree']]

if len(disagreements) == 0:
    print("✅ Perfect agreement on dev set!")
else:
    print(f"Found {len(disagreements)} disagreement(s):\n")
    
    for _, row in disagreements.iterrows():
        print(f"Question: {row['question']}")
        print(f"  Model response: {row['model_response'][:80]}...")
        print(f"  Human label:    {row['human_label']}")
        print(f"  Judge label:    {row['judge_label']}")
        print(f"  Human reason:   {row['human_rationale'][:100]}...")
        print(f"  Judge reason:   {row['judge_reasoning'][:100]}...")
        print()

## 7. Iterate on the Judge Prompt

Based on the disagreement analysis above, we can refine the judge prompt. Common refinements:

- **Clarify borderline cases** — if the judge is too strict on approximations, add guidance
- **Add edge case handling** — if the judge mishandles comparisons or refusals, add examples
- **Sharpen the rubric** — if the judge conflates criteria, narrow the definition

After refining, re-run on the dev set to measure improvement. **Never look at the test set during this process.**

Below we create a refined prompt and re-evaluate:

In [ ]:
# Refined judge prompt — adjust based on disagreement patterns observed above
JUDGE_PROMPT_V2 = """You are evaluating whether a model's response contains factually accurate data compared to the provided context.

## Evaluation Criterion: Factual Accuracy

### PASS when:
- The numerical data in the response matches the context exactly
- Reasonable approximations are used (e.g., "about 2.4 million" for 2,390,125)
- The response includes additional true commentary beyond what's in the context
- Minor formatting differences (e.g., 518 vs 518.0)

### FAIL when:
- The response states a specific number that differs from the context (even if close)
- The response reverses a comparison (says A > B when B > A)
- The response claims data is unavailable when the context provides it
- The response does not answer the actual question asked
- The response rounds to a number that misrepresents the data (e.g., 887,642 → 900,000)

### Important edge cases:
- "Approximately 2.4 million" for 2,390,125 → PASS (reasonable rounding)
- "920,000" for 918,915 → FAIL (specific number that's wrong)
- Stating correct data plus unverifiable claims → PASS (judge only the factual data)

{few_shot_section}

## Now evaluate this response:

**Question:** {question}
**Context:** {context}
**Model Response:** {model_response}

Respond with ONLY this JSON (no other text):
{{
    "reasoning": "<your step-by-step analysis>",
    "label": "<pass or fail>"
}}
"""

# Temporarily swap the template
original_template = JUDGE_PROMPT_TEMPLATE
JUDGE_PROMPT_TEMPLATE = JUDGE_PROMPT_V2

print("Running refined judge (v2) on dev set...")
dev_results_v2 = evaluate_split(dev_set, fewshot_set, "dev-v2")

# Restore original for comparison
JUDGE_PROMPT_TEMPLATE = original_template

# Calculate v2 metrics
df_dev_v2 = pd.DataFrame(dev_results_v2)

actual_pass_v2 = df_dev_v2[df_dev_v2['human_label'] == 'pass']
tp_v2 = len(actual_pass_v2[actual_pass_v2['judge_label'] == 'pass'])
tpr_v2 = tp_v2 / len(actual_pass_v2) if len(actual_pass_v2) > 0 else 0

actual_fail_v2 = df_dev_v2[df_dev_v2['human_label'] == 'fail']
tn_v2 = len(actual_fail_v2[actual_fail_v2['judge_label'] == 'fail'])
tnr_v2 = tn_v2 / len(actual_fail_v2) if len(actual_fail_v2) > 0 else 0

accuracy_v2 = df_dev_v2['agree'].mean()

print(f"\n{'='*50}")
print(f"Dev Set Comparison")
print(f"{'='*50}")
print(f"{'Metric':<25} {'v1':>10} {'v2':>10}")
print(f"{'-'*50}")
print(f"{'Overall accuracy':<25} {overall_accuracy:>9.1%} {accuracy_v2:>9.1%}")
print(f"{'TPR (real passes)':<25} {tpr:>9.1%} {tpr_v2:>9.1%}")
print(f"{'TNR (real fails)':<25} {tnr:>9.1%} {tnr_v2:>9.1%}")
print(f"{'='*50}")

# Use the better version going forward
if accuracy_v2 >= overall_accuracy:
    JUDGE_PROMPT_TEMPLATE = JUDGE_PROMPT_V2
    print("\n→ Using v2 (refined) for final validation")
else:
    print("\n→ Keeping v1 (original) for final validation")

In [ ]:
# Calibration Improvement — v1 vs v2
fig, ax = plt.subplots(figsize=(8, 5))

metrics = ['Overall\nAccuracy', 'TPR\n(Real Passes)', 'TNR\n(Real Fails)']
v1_scores = [overall_accuracy, tpr, tnr]
v2_scores = [accuracy_v2, tpr_v2, tnr_v2]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, v1_scores, width, label='v1 (Original)', color='#4C72B0', alpha=0.8)
bars2 = ax.bar(x + width/2, v2_scores, width, label='v2 (Refined)', color='#55A868', alpha=0.8)

ax.set_ylabel('Score')
ax.set_title('Judge Calibration: v1 vs v2 Prompt')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15)
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.0%}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 8. Final Validation on Test Set

Now we run the calibrated judge on the **held-out test set** that was never seen during development. This gives us the unbiased estimate of judge accuracy.

⚠️ **If test performance is unsatisfactory, collect new labeled data and repeat the process. Do not re-tune on the test set.**

In [ ]:
print("Running final validation on test set...")
test_results = evaluate_split(test_set, fewshot_set, "test")

df_test = pd.DataFrame(test_results)

actual_pass_test = df_test[df_test['human_label'] == 'pass']
tp_test = len(actual_pass_test[actual_pass_test['judge_label'] == 'pass'])
tpr_test = tp_test / len(actual_pass_test) if len(actual_pass_test) > 0 else 0

actual_fail_test = df_test[df_test['human_label'] == 'fail']
tn_test = len(actual_fail_test[actual_fail_test['judge_label'] == 'fail'])
tnr_test = tn_test / len(actual_fail_test) if len(actual_fail_test) > 0 else 0

accuracy_test = df_test['agree'].mean()

print(f"\n{'='*50}")
print(f"FINAL TEST SET RESULTS (n={len(test_results)})")
print(f"{'='*50}")
print(f"Overall accuracy: {accuracy_test:.1%}")
print(f"TPR (catches real passes):  {tpr_test:.1%}  ({tp_test}/{len(actual_pass_test)})")
print(f"TNR (catches real fails):   {tnr_test:.1%}  ({tn_test}/{len(actual_fail_test)})")
print(f"{'='*50}")

# Show any test disagreements
test_disagreements = df_test[~df_test['agree']]
if len(test_disagreements) > 0:
    print(f"\nTest set disagreements ({len(test_disagreements)}):")
    for _, row in test_disagreements.iterrows():
        print(f"  • {row['question'][:60]}  human={row['human_label']}  judge={row['judge_label']}")

## 9. Test Judge Repeatability

Even with default sampling parameters, LLM outputs can vary slightly across runs. A reliable judge should produce the **same verdict on the same input every time**.

We run a subset of examples multiple times and check for label flips.

In [ ]:
# Pick a subset for repeatability testing
repeat_subset = dev_set[:5]  # First 5 dev examples
N_RUNS = 3

print(f"Running {N_RUNS} repeated evaluations on {len(repeat_subset)} examples...\n")

repeat_results = {ex['question']: [] for ex in repeat_subset}

for run in range(N_RUNS):
    print(f"  Run {run + 1}/{N_RUNS}...")
    for ex in repeat_subset:
        result = run_judge(ex['question'], ex['context'], ex['model_response'], fewshot_set)
        repeat_results[ex['question']].append(result['label'])
        sleep(0.5)

# Analyze stability
print(f"\n{'='*50}")
print(f"Repeatability Results ({N_RUNS} runs)")
print(f"{'='*50}")

stable_count = 0
for question, labels in repeat_results.items():
    is_stable = len(set(labels)) == 1
    stable_count += is_stable
    status = "✅ Stable" if is_stable else "⚠️ FLIPPED"
    print(f"  {status}: {question[:50]}...")
    if not is_stable:
        print(f"           Labels across runs: {labels}")

repeatability = stable_count / len(repeat_subset)
print(f"\nRepeatability: {repeatability:.0%} ({stable_count}/{len(repeat_subset)} stable)")

In [ ]:
# Repeatability Visualization
fig, ax = plt.subplots(figsize=(10, 4))

questions_short = [q[:35] + '...' for q in repeat_results.keys()]
label_map = {'pass': 1, 'fail': 0, 'error': -1}

for i, (question, labels) in enumerate(repeat_results.items()):
    numeric_labels = [label_map.get(l, -1) for l in labels]
    is_stable = len(set(labels)) == 1
    color = '#55A868' if is_stable else '#C44E52'
    
    for run, val in enumerate(numeric_labels):
        ax.scatter(run, i, c=color, s=100, zorder=3, edgecolors='white', linewidth=0.5)

ax.set_xticks(range(N_RUNS))
ax.set_xticklabels([f'Run {r+1}' for r in range(N_RUNS)])
ax.set_yticks(range(len(questions_short)))
ax.set_yticklabels(questions_short, fontsize=9)
ax.set_title('Judge Repeatability Across Runs')
ax.set_xlabel('Run')

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#55A868', markersize=10, label='Stable'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#C44E52', markersize=10, label='Flipped')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

## 10. Multiple Failure Modes

Each judge should evaluate **one specific failure mode**. To cover multiple aspects of quality, build separate judges and combine results. Here we add a second judge for **completeness** — does the response actually answer what was asked?

In [ ]:
# Completeness Judge — separate from factual accuracy
COMPLETENESS_JUDGE_PROMPT = """You are evaluating whether a model's response completely answers the question asked.

## Evaluation Criterion: Completeness

### PASS when:
- The response addresses all parts of the question
- For comparison questions, both items are discussed with data
- For multi-part questions, all parts are answered

### FAIL when:
- The response only answers part of a multi-part question
- The response provides related but different information than what was asked
- The response is vague or generic without answering the specific question
- For comparison questions, only one item is discussed or no numbers are given

{few_shot_section}

## Now evaluate this response:

**Question:** {question}
**Context:** {context}
**Model Response:** {model_response}

Respond with ONLY this JSON (no other text):
{{
    "reasoning": "<your step-by-step analysis>",
    "label": "<pass or fail>"
}}
"""

# Get completeness examples from benchmark
completeness_examples = [ex for ex in benchmark if ex['failure_mode'] == 'completeness']
comp_fewshot = completeness_examples[:2]
comp_eval = completeness_examples[2:]

print(f"Completeness judge: {len(comp_fewshot)} few-shot, {len(comp_eval)} to evaluate")

# Run completeness judge
original_template = JUDGE_PROMPT_TEMPLATE
JUDGE_PROMPT_TEMPLATE = COMPLETENESS_JUDGE_PROMPT

print("Running completeness judge...")
comp_results = evaluate_split(comp_eval, comp_fewshot, "completeness")
JUDGE_PROMPT_TEMPLATE = original_template

df_comp = pd.DataFrame(comp_results)
comp_accuracy = df_comp['agree'].mean()
print(f"\nCompleteness judge accuracy: {comp_accuracy:.0%}")

In [ ]:
# Combined Multi-Judge Results
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: per-judge accuracy
judges = ['Factual\nAccuracy', 'Completeness']
accuracies = [accuracy_test, comp_accuracy]
colors = ['#4C72B0', '#55A868']
bars = axes[0].bar(judges, accuracies, color=colors, width=0.5)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Per-Judge Accuracy')
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{acc:.0%}', ha='center', fontweight='bold')

# Right: combined pass/fail per example (for completeness eval set)
if len(comp_results) > 0:
    labels_list = ['Both Pass', 'Accuracy Pass\nCompleteness Fail', 
                   'Accuracy Fail\nCompleteness Pass', 'Both Fail']
    # For examples that appear in both evaluations, show combined
    combined_counts = [0, 0, 0, 0]
    for r in comp_results:
        acc_label = r.get('human_label', 'unknown')  # Using human label as proxy
        comp_label = r.get('judge_label', 'unknown')
        if acc_label == 'pass' and comp_label == 'pass':
            combined_counts[0] += 1
        elif acc_label == 'pass' and comp_label == 'fail':
            combined_counts[1] += 1
        elif acc_label == 'fail' and comp_label == 'pass':
            combined_counts[2] += 1
        else:
            combined_counts[3] += 1
    
    colors_pie = ['#55A868', '#FFBB78', '#AEC7E8', '#C44E52']
    non_zero = [(l, c, col) for l, c, col in zip(labels_list, combined_counts, colors_pie) if c > 0]
    if non_zero:
        axes[1].pie([x[1] for x in non_zero], labels=[x[0] for x in non_zero],
                   colors=[x[2] for x in non_zero], autopct='%1.0f%%', startangle=90)
    axes[1].set_title('Combined Judge Verdicts')

plt.tight_layout()
plt.show()

print("\n💡 Each judge evaluates ONE failure mode independently.")
print("   Combine results to get a multi-dimensional quality picture.")

## 11. Judge Bias Check

A reliable judge should not be swayed by surface-level features like response length, confident language, or formatting. We test this by creating perturbations of the same response and checking if the judge's verdict changes.

In [ ]:
# Bias Check — test if the judge is swayed by surface features
bias_test_cases = [
    {
        "name": "Verbosity bias",
        "question": "What is the population of Dallas?",
        "context": "Dallas, TX: population=1,343,573, land_area_mi2=339.6",
        "original": "Dallas has a population of 1,343,573.",
        "perturbed": "Dallas, a major city in the state of Texas and one of the most important economic centers in the southern United States, has a population of 1,343,573 according to the most recent available census data, which reflects the city's continued growth and development over the past several decades."
    },
    {
        "name": "Confidence bias",
        "question": "What is the population of Seattle?",
        "context": "Seattle, WA: population=737,015, land_area_mi2=83.8",
        "original": "Seattle has a population of 750,000.",
        "perturbed": "I am absolutely certain that Seattle has a population of 750,000. This is a well-established fact that I can confirm with complete confidence."
    },
    {
        "name": "Formatting bias",
        "question": "What is the population of Austin?",
        "context": "Austin, TX: population=978,908, land_area_mi2=319.9",
        "original": "Austin has a population of 978,908.",
        "perturbed": "## Austin, Texas\n\n**Population:** 978,908\n\n*Source: US Census Data*"
    }
]

print("Running bias checks...\n")
bias_results = []

for test in bias_test_cases:
    orig_result = run_judge(test['question'], test['context'], test['original'], fewshot_set)
    sleep(0.5)
    pert_result = run_judge(test['question'], test['context'], test['perturbed'], fewshot_set)
    sleep(0.5)
    
    flipped = orig_result['label'] != pert_result['label']
    bias_results.append({
        'name': test['name'],
        'original_label': orig_result['label'],
        'perturbed_label': pert_result['label'],
        'flipped': flipped
    })
    
    status = "⚠️ BIAS DETECTED" if flipped else "✅ Stable"
    print(f"{status} — {test['name']}")
    print(f"  Original:  {test['original'][:60]}... → {orig_result['label']}")
    print(f"  Perturbed: {test['perturbed'][:60]}... → {pert_result['label']}")
    print()

# Bias summary
biased = sum(1 for r in bias_results if r['flipped'])
print(f"Bias check: {biased}/{len(bias_results)} tests showed label flip")
if biased == 0:
    print("✅ Judge appears robust to surface-level perturbations")
else:
    print("⚠️ Judge may be influenced by response style rather than content")

## 12. Next Step: Discovering Failure Modes (Open Coding)

This notebook calibrated judges for failure modes we already knew about (factual accuracy, completeness). But where does the list of failure modes come from in the first place? The answer is **open coding**: reading traces, writing brief notes about what went wrong, and clustering them into categories — letting the data tell you what's failing rather than starting from a predetermined list.

That process is its own foundational module. Continue with [03: Understanding Failures](../03-understanding-failures/01_Discovering_Failure_Patterns.ipynb), which walks through open coding on real agent traces, prioritizes the discovered problems by impact, and bridges them into judge prompts like the ones you built here.


## 13. Judge Scorecard

A summary dashboard of judge quality metrics. Use this to decide whether the judge is ready for production use.

In [ ]:
print("=" * 60)
print("  JUDGE SCORECARD")
print("=" * 60)
print()
print(f"  Failure Mode:        Factual Accuracy")
print(f"  Judge Model:         {JUDGE_MODEL}")
print(f"  Benchmark Size:      {len(benchmark)} examples")
print()
print(f"  ┌─────────────────────────────────────────┐")
print(f"  │ Dev Set Accuracy:    {accuracy_v2:.0%}                │")
print(f"  │ Test Set Accuracy:   {accuracy_test:.0%}                │")
print(f"  │ TPR (test):          {tpr_test:.0%}                │")
print(f"  │ TNR (test):          {tnr_test:.0%}                │")
print(f"  │ Repeatability:       {repeatability:.0%}                │")
print(f"  │ Bias Checks Passed: {len(bias_results) - biased}/{len(bias_results)}                │")
print(f"  │ Completeness Judge:  {comp_accuracy:.0%}                │")
print(f"  └─────────────────────────────────────────┘")
print()

# Recommendations
print("  Recommendations:")
if accuracy_test >= 0.9 and repeatability >= 0.8:
    print("  ✅ Judge is suitable for automated evaluation")
elif accuracy_test >= 0.75:
    print("  ⚠️ Judge is usable but consider human review for borderline cases")
else:
    print("  ❌ Judge needs further calibration before production use")

if repeatability < 0.8:
    print("  ⚠️ Low repeatability — consider adding more few-shot examples or using system-prompt instructions to guide behavior")

if biased > 0:
    print("  ⚠️ Judge shows bias toward surface features — review the bias check results")

if tpr_test < tnr_test - 0.2:
    print("  ⚠️ Judge is too strict — missing real passes. Relax the rubric for borderline cases.")
elif tnr_test < tpr_test - 0.2:
    print("  ⚠️ Judge is too lenient — missing real fails. Tighten the rubric.")

print()
print("=" * 60)

## Key Takeaways

1. **Evaluate your judge before trusting it.** A judge is just another model — it needs its own test set.

2. **Binary pass/fail beats rating scales.** Scales introduce implicit variation that hides failure modes. A series of focused pass/fail tests is far more useful.

3. **One failure mode per judge.** Don't ask a single judge to check accuracy, completeness, and tone. Build separate judges and combine results.

4. **Split your labeled data.** Few-shot examples come from the training split. Iterate on the dev split. Validate on the test split. Never contaminate.

5. **Measure TPR and TNR, not just accuracy.** A judge that always says "pass" has 100% TPR but 0% TNR. You need both.

6. **Test repeatability.** If the judge flips its verdict on the same input, you can't trust it.

7. **Discovering failure modes is its own discipline.** Open coding — reading traces and letting the data reveal failure categories — is covered in module 03: Understanding Failures.

### What's Next?

- Continue with [03: Understanding Failures](../03-understanding-failures/01_Discovering_Failure_Patterns.ipynb) to discover failure modes from real traces via open coding
- Add more failure modes (completeness, reasoning quality) as separate judges
- Scale the benchmark to 100+ examples per failure mode
- Integrate judge validation into your CI/CD pipeline